# Problem Set 2: Neural networks & optimisation

Analysis of Unstructured Data. Author: Rafał Głodek, Bartłomiej Mielcarz

Problems 1–3 are theory answers in Markdown. Problems 4–7 call the code in `src/` and discuss the results.

---

## Problem 1 : Softmax + Cross-Entropy

### **Setup**

We are given the logits $z\in\mathbb{R}^K$, the softmax probabilities

$$
p_k=\frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}},
$$

and the cross-entropy loss

$$
L=-\sum_{k=1}^K y_k\log p_k,
$$

where $y$ is a one-hot encoded target, meaning

$$
y_k\in\{0,1\},\qquad \sum_{k=1}^K y_k=1.
$$

We want to prove that

$$
\boxed{\frac{\partial L}{\partial z_i}=p_i-y_i.}
$$

### **Step 1: substitute the softmax into the loss**

Taking the logarithm of $p_k$, we obtain

$$
\begin{aligned}
\log p_k
&=\log\left(\frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}\right)\\
&=z_k-\log\left(\sum_{j=1}^K e^{z_j}\right).
\end{aligned}
$$

Substituting into the cross-entropy loss,

$$
\begin{aligned}
L
&=-\sum_{k=1}^K y_k
\left(z_k-\log\left(\sum_{j=1}^K e^{z_j}\right)\right)\\[4pt]
&=-\sum_{k=1}^K y_kz_k
+\left(\sum_{k=1}^K y_k\right)
\log\left(\sum_{j=1}^K e^{z_j}\right).
\end{aligned}
$$

Since $\sum_{k=1}^K y_k=1$, we get

$$
\boxed{
L=-\sum_{k=1}^K y_kz_k+
\log\left(\sum_{j=1}^K e^{z_j}\right).
}
$$

### **Step 2: differentiate with respect to $z_i$**

Differentiating both terms separately,

$$
\frac{\partial L}{\partial z_i}
=
-\frac{\partial}{\partial z_i}
\left(\sum_{k=1}^K y_kz_k\right)
+
\frac{\partial}{\partial z_i}
\log\left(\sum_{j=1}^K e^{z_j}\right).
$$

For the first term,

$$
-\frac{\partial}{\partial z_i}
\left(\sum_{k=1}^K y_kz_k\right)=-y_i.
$$

For the second term, applying the chain rule,

$$
\begin{aligned}
\frac{\partial}{\partial z_i}
\log\left(\sum_{j=1}^K e^{z_j}\right)
&=\frac{1}{\sum_{j=1}^K e^{z_j}}
\frac{\partial}{\partial z_i}
\left(\sum_{j=1}^K e^{z_j}\right)\\[4pt]
&=\frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}\\[4pt]
&=p_i.
\end{aligned}
$$

Combining the two results,

$$
\boxed{\frac{\partial L}{\partial z_i}=p_i-y_i.}
$$

Equivalently, in vector notation,

$$
\boxed{\nabla_z L=p-y.}\qquad\blacksquare
$$

### **Step 3: why are softmax and cross-entropy paired?**

The key advantage is that **their combination produces an exceptionally simple gradient**.

Instead of explicitly computing the full softmax Jacobian, the gradient reduces to the difference between the predicted probabilities and the target distribution:

$$
\nabla_z L=p-y.
$$

This has several benefits:

1. **Simple computation:** The gradient requires only subtracting the target vector from the predicted probability vector.

2. **Clear interpretation:** For the correct class $i$, the gradient is $p_i-1$, encouraging its logit to increase. For incorrect classes, the gradient is $p_i$, encouraging their logits to decrease during gradient descent.

3. **Numerical stability:** The combined loss can be implemented using a stable log-sum-exp expression, avoiding explicitly computing $\log p_i$ for extremely small probabilities.

4. **Effective backpropagation:** The simplified gradient avoids an additional multiplicative softmax derivative that could make gradients unnecessarily small.

---

## Problem 2: Backpropagation in a Two-Layer MLP

We are given a two-layer neural network:

$$
h=\sigma(W_1x+b_1),
$$

$$
\hat y=\operatorname{softmax}(W_2h+b_2),
$$

with the cross-entropy loss

$$
L=-\sum_{k=1}^K y_k\log\hat y_k.
$$

Our goal is to derive

$$
\boxed{
\frac{\partial L}{\partial W_2},\quad
\frac{\partial L}{\partial b_2},\quad
\frac{\partial L}{\partial W_1},\quad
\frac{\partial L}{\partial b_1}.
}
$$

### **Step 1: define intermediate variables**

To simplify the notation, let

$$
\begin{aligned}
z_1 &= W_1x+b_1,\\
h &= \sigma(z_1),\\
z_2 &= W_2h+b_2,\\
\hat y &= \operatorname{softmax}(z_2).
\end{aligned}
$$

Assume the dimensions are

$$
\begin{aligned}
x&\in\mathbb R^d,\\
W_1&\in\mathbb R^{m\times d},\\
b_1&\in\mathbb R^m,\\
W_2&\in\mathbb R^{K\times m},\\
b_2&\in\mathbb R^K.
\end{aligned}
$$

The computational graph is

$$
x \longrightarrow z_1 \longrightarrow h
\longrightarrow z_2 \longrightarrow \hat y
\longrightarrow L.
$$

Backpropagation proceeds in the opposite direction.

### **Step 2: gradient with respect to the output layer**

From Problem 1, we already know that softmax combined with cross-entropy gives

$$
\boxed{\frac{\partial L}{\partial z_2}=\hat y-y.}
$$

Define the output error:

$$
\delta_2=\hat y-y.
$$

#### **2.1. Derivative with respect to $W_2$**

Recall that

$$
z_{2,k}=\sum_{j=1}^m W_{2,kj}h_j+b_{2,k}.
$$

Applying the chain rule,

$$
\frac{\partial L}{\partial W_{2,kj}}
=
\sum_{\ell=1}^K
\frac{\partial L}{\partial z_{2,\ell}}
\frac{\partial z_{2,\ell}}{\partial W_{2,kj}}.
$$

Since

$$
\frac{\partial z_{2,\ell}}{\partial W_{2,kj}}
=
\begin{cases}
h_j,&\ell=k,\\
0,&\ell\neq k,
\end{cases}
$$

we obtain

$$
\frac{\partial L}{\partial W_{2,kj}}
=(\hat y_k-y_k)h_j.
$$

Therefore, in matrix notation,

$$
\boxed{
\frac{\partial L}{\partial W_2}
=(\hat y-y)h^\top.
}
$$

#### **2.2. Derivative with respect to $b_2$**

Similarly,

$$
\frac{\partial L}{\partial b_{2,k}}
=
\sum_{\ell=1}^K
\frac{\partial L}{\partial z_{2,\ell}}
\frac{\partial z_{2,\ell}}{\partial b_{2,k}}.
$$

Because

$$
\frac{\partial z_{2,\ell}}{\partial b_{2,k}}
=
\begin{cases}
1,&\ell=k,\\
0,&\ell\neq k,
\end{cases}
$$

we get

$$
\boxed{
\frac{\partial L}{\partial b_2}
=\hat y-y.
}
$$

### **Step 3: backpropagate through the hidden layer**

We now calculate the gradient of $L$ with respect to $z_1$.

First, using the chain rule,

$$
\frac{\partial L}{\partial h_j}
=
\sum_{k=1}^K
\frac{\partial L}{\partial z_{2,k}}
\frac{\partial z_{2,k}}{\partial h_j}.
$$

Since

$$
z_{2,k}=\sum_{j=1}^m W_{2,kj}h_j+b_{2,k},
$$

we have

$$
\frac{\partial z_{2,k}}{\partial h_j}=W_{2,kj}.
$$

Thus,

$$
\frac{\partial L}{\partial h_j}
=
\sum_{k=1}^K(\hat y_k-y_k)W_{2,kj}.
$$

In vector notation,

$$
\boxed{
\frac{\partial L}{\partial h}
=W_2^\top(\hat y-y).
}
$$

Next, recall that

$$
h_j=\sigma(z_{1,j}).
$$

Because $\sigma$ acts elementwise,

$$
\frac{\partial h_j}{\partial z_{1,j}}
=\sigma'(z_{1,j}).
$$

Applying the chain rule once more,

$$
\frac{\partial L}{\partial z_{1,j}}
=
\frac{\partial L}{\partial h_j}
\sigma'(z_{1,j}).
$$

Therefore,

$$
\boxed{
\frac{\partial L}{\partial z_1}
=
\left(W_2^\top(\hat y-y)\right)
\odot\sigma'(z_1).
}
$$

Here, $\odot$ denotes elementwise multiplication (Hadamard product).

Define the hidden-layer error:

$$
\boxed{
\delta_1=
\left(W_2^\top\delta_2\right)\odot\sigma'(z_1).
}
$$

### **Step 4: gradient with respect to the first layer**

#### **4.1. Derivative with respect to $W_1$**

Recall that

$$
z_{1,j}=\sum_{i=1}^d W_{1,ji}x_i+b_{1,j}.
$$

By the chain rule,

$$
\frac{\partial L}{\partial W_{1,ji}}
=
\sum_{\ell=1}^m
\frac{\partial L}{\partial z_{1,\ell}}
\frac{\partial z_{1,\ell}}{\partial W_{1,ji}}.
$$

Since

$$
\frac{\partial z_{1,\ell}}{\partial W_{1,ji}}
=
\begin{cases}
x_i,&\ell=j,\\
0,&\ell\neq j,
\end{cases}
$$

we obtain

$$
\frac{\partial L}{\partial W_{1,ji}}
=\delta_{1,j}x_i.
$$

Therefore,

$$
\boxed{
\frac{\partial L}{\partial W_1}
=
\delta_1x^\top
=
\left[
W_2^\top(\hat y-y)\odot\sigma'(z_1)
\right]x^\top.
}
$$

#### **4.2. Derivative with respect to $b_1$**

Similarly,

$$
\frac{\partial L}{\partial b_{1,j}}
=
\frac{\partial L}{\partial z_{1,j}}
\frac{\partial z_{1,j}}{\partial b_{1,j}}.
$$

Since

$$
\frac{\partial z_{1,j}}{\partial b_{1,j}}=1,
$$

we obtain

$$
\boxed{
\frac{\partial L}{\partial b_1}
=
\delta_1
=
W_2^\top(\hat y-y)\odot\sigma'(z_1).
}
$$

### **Final result**

Let

$$
\delta_2=\hat y-y,
$$

$$
\delta_1=(W_2^\top\delta_2)\odot\sigma'(W_1x+b_1).
$$

Then all four gradients are:

**Output layer:**

$$
\boxed{
\frac{\partial L}{\partial W_2}=\delta_2h^\top
}
$$

$$
\boxed{
\frac{\partial L}{\partial b_2}=\delta_2
}
$$

**Hidden layer:**

$$
\boxed{
\frac{\partial L}{\partial W_1}=\delta_1x^\top
}
$$

$$
\boxed{
\frac{\partial L}{\partial b_1}=\delta_1
}
$$

These formulas hold for any differentiable elementwise activation function $\sigma$. At nondifferentiable points, such as zero for ReLU, a conventional subgradient may be used.


---
## Problem 3: SGD, Momentum and Adam

We want to:

1. Write the update rules for **SGD, SGD with momentum, and Adam**.
2. Show that momentum is an **exponential moving average (EMA)** of gradients.
3. Explain why Adam uses **bias correction** in its first few iterations.

### **Step 1: Stochastic Gradient Descent (SGD)**

Let $\theta\in\mathbb R^d$ denote the model parameters, and let $L(\theta)$ be the loss function.

At iteration $t$, we compute the stochastic gradient

$$
g_t=\nabla_\theta L_{\mathcal B_t}(\theta_{t-1}),
$$

where $\mathcal B_t$ denotes the sampled mini-batch.

The SGD update rule is

$$
\boxed{\theta_t=\theta_{t-1}-\eta g_t}
$$

where $\eta>0$ is the learning rate.

SGD moves the parameters in the direction opposite to the gradient. However, stochastic gradients can be noisy, causing oscillations during optimization.

### **Step 2: SGD with Momentum**

Momentum introduces an auxiliary variable $m_t$, which accumulates information from previous gradients.

Using the normalized momentum convention, the update rules are

$$
\boxed{
\begin{aligned}
m_t&=\beta m_{t-1}+(1-\beta)g_t,\\
\theta_t&=\theta_{t-1}-\eta m_t,
\end{aligned}
}
$$

where

- $\beta\in[0,1)$ is the momentum coefficient, typically $0.9$.
- $m_0=0$ is the initial momentum.
- $\eta$ is the learning rate.

#### **2.1. Show that momentum is an exponential moving average**

Starting with

$$
m_t=\beta m_{t-1}+(1-\beta)g_t,
$$

we recursively substitute earlier values.

For $t=1$,

$$
m_1=\beta m_0+(1-\beta)g_1.
$$

Since $m_0=0$,

$$
m_1=(1-\beta)g_1.
$$

For $t=2$,

$$
\begin{aligned}
m_2
&=\beta m_1+(1-\beta)g_2\\
&=\beta(1-\beta)g_1+(1-\beta)g_2.
\end{aligned}
$$

For $t=3$,

$$
\begin{aligned}
m_3
&=\beta m_2+(1-\beta)g_3\\
&=\beta^2(1-\beta)g_1
+\beta(1-\beta)g_2
+(1-\beta)g_3.
\end{aligned}
$$

Continuing this process gives

$$
\boxed{
m_t=(1-\beta)\sum_{i=1}^{t}\beta^{t-i}g_i.
}
$$

Thus, momentum is an **exponentially weighted sum of past gradients**.


### **Step 3: Adam (Adaptive Moment Estimation)**

Adam combines momentum with adaptive learning rates.

It maintains two moving averages:

- $m_t$: exponential moving average of gradients (first moment).
- $v_t$: exponential moving average of squared gradients (second raw moment).

The update rules are:

**1. Compute the gradient**

$$
g_t=\nabla_\theta L_{\mathcal B_t}(\theta_{t-1}).
$$

**2. Update the first moment**

$$
\boxed{
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
}
$$

**3. Update the second moment**

$$
\boxed{
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
}
$$

where $g_t^2$ denotes elementwise squaring.

**4. Apply bias correction**

$$
\boxed{
\hat m_t=\frac{m_t}{1-\beta_1^t}
}
$$

$$
\boxed{
\hat v_t=\frac{v_t}{1-\beta_2^t}
}
$$

**5. Update the parameters**

$$
\boxed{
\theta_t=
\theta_{t-1}
-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}
}
$$

All operations involving vectors in the final update are performed elementwise.

Typical hyperparameters are

$$
\beta_1=0.9,\qquad
\beta_2=0.999,\qquad
\varepsilon=10^{-8}.
$$

We initialize $m_0=v_0=0$.

### **Step 4: why does Adam need bias correction?**

Both moving averages start at zero. Consequently, during the first few iterations, their values are systematically biased toward zero.

Consider the first moment:

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t.
$$

From Step 2, we know that

$$
m_t=(1-\beta_1)\sum_{i=1}^t\beta_1^{t-i}g_i.
$$

Assume that the gradients have a constant expected value

$$
\mathbb E[g_i]=\mu.
$$

Taking expectations,

$$
\begin{aligned}
\mathbb E[m_t]
&=(1-\beta_1)
\sum_{i=1}^t\beta_1^{t-i}\mathbb E[g_i]\\
&=(1-\beta_1)\mu
\sum_{i=1}^t\beta_1^{t-i}.
\end{aligned}
$$

Using the geometric series formula,

$$
\sum_{i=1}^t\beta_1^{t-i}
=\frac{1-\beta_1^t}{1-\beta_1},
$$

we obtain

$$
\boxed{
\mathbb E[m_t]=(1-\beta_1^t)\mu.
}
$$

Since $1-\beta_1^t<1$, the estimate is biased toward zero.

To correct this, Adam defines

$$
\hat m_t=\frac{m_t}{1-\beta_1^t}.
$$

Therefore,

$$
\begin{aligned}
\mathbb E[\hat m_t]
&=\frac{\mathbb E[m_t]}{1-\beta_1^t}\\
&=\frac{(1-\beta_1^t)\mu}{1-\beta_1^t}\\
&=\mu.
\end{aligned}
$$

Hence,

$$
\boxed{\mathbb E[\hat m_t]=\mu.}
$$

Under the constant-moment assumption, the bias-corrected first moment is unbiased.

The same argument applies to the second moment. If

$$
\mathbb E[g_i^2]=\nu,
$$

then

$$
\mathbb E[v_t]=(1-\beta_2^t)\nu.
$$

Consequently,

$$
\boxed{
\hat v_t=\frac{v_t}{1-\beta_2^t}
}
$$

removes this initialization bias.

#### **4.1. Example: the first iteration**

Consider Adam with

$$
\beta_1=0.9,\qquad \beta_2=0.999.
$$

At $t=1$, since $m_0=v_0=0$,

$$
m_1=0.1g_1
$$

and

$$
v_1=0.001g_1^2.
$$

Both estimates are much smaller than the gradient quantities they are intended to estimate.

After bias correction,

$$
\begin{aligned}
\hat m_1
&=\frac{0.1g_1}{1-0.9}\\
&=g_1,
\end{aligned}
$$

and

$$
\begin{aligned}
\hat v_1
&=\frac{0.001g_1^2}{1-0.999}\\
&=g_1^2.
\end{aligned}
$$

Thus,

$$
\boxed{
\hat m_1=g_1,\qquad \hat v_1=g_1^2.
}
$$

Bias correction prevents the initial zero values from artificially shrinking the moment estimates. This is particularly important for the second moment, since $\beta_2$ is typically very close to 1.

As $t\to\infty$,

$$
\beta_1^t\to0,\qquad \beta_2^t\to0,
$$

so

$$
\hat m_t\approx m_t,\qquad
\hat v_t\approx v_t.
$$

Therefore, bias correction matters most during the first few iterations.

## Problem 4: Two-layer MLP in NumPy with a gradient check

## Problem 5: The same network in PyTorch

## Problem 6: Regularisation

## Problem 7: Over- and under-fitting